# Task 2 — Step 2: PPO Policy Clipping Study ($\epsilon$ Ablation)

### Research Question
How does the clipping parameter $\epsilon \in \{0.05, 0.20, 0.50\}$ alter:
1. **Geometric constraint**: Fraction of tokens clipped and affected-token fraction on the fixed cached rollout batch (`cached/ppo_rollout.pt`).
2. **Optimization stability**: Short-fork continuation stability, gradient norm variance, and held-out response quality under matched budgets.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run Clipping Study or Load Results


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task2_ppo.analyze_clipping --config configs/ppo.yaml
else:
    print("[OK] Loading saved clipping study results...")


## 2. Geometric Clipping Analysis on Cached Batch


In [ ]:
clip_file = "results/task2_ppo/ppo_clipping_study_results.json"
if os.path.exists(clip_file):
    with open(clip_file, encoding='utf-8') as f:
        clip_data = json.load(f)
    
    cached = clip_data.get("cached_geometry", {})
    forks = clip_data.get("fork_results", {})
    
    rows = []
    for eps_str in sorted(cached.keys(), key=float):
        c_stats = cached[eps_str]
        f_eval = forks.get(eps_str, {}).get("eval", {})
        rows.append({
            "Epsilon (ε)": float(eps_str),
            "Cached Clip Frac (%)": f"{c_stats.get('clip_fraction', 0)*100:.2f}%",
            "Affected Token Frac (%)": f"{c_stats.get('affected_fraction', 0)*100:.2f}%",
            "Cached Surrogate Loss": f"{c_stats.get('surrogate_loss', 0):.4f}",
            "Fork Mean Reward": f"{f_eval.get('mean_reward', float('nan')):.3f}",
            "Fork Mean KL": f"{f_eval.get('mean_kl', float('nan')):.4f}"
        })
    df_clip = pd.DataFrame(rows).sort_values("Epsilon (ε)")
    display(df_clip)


## 3. Visualizing Token Clipping vs. Epsilon


In [ ]:
if os.path.exists(clip_file):
    eps_vals = df_clip["Epsilon (ε)"].values
    clip_pct = [float(r["Cached Clip Frac (%)"].replace('%','')) for _, r in df_clip.iterrows()]
    aff_pct = [float(r["Affected Token Frac (%)"].replace('%','')) for _, r in df_clip.iterrows()]
    
    x = np.arange(len(eps_vals))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, clip_pct, width, label='Clipped Tokens (|ρ - 1| > ε)', color='#EF4444', alpha=0.85)
    ax.bar(x + width/2, aff_pct, width, label='Affected Tokens', color='#F59E0B', alpha=0.85)
    ax.set_ylabel('Percentage of Tokens (%)')
    ax.set_title('PPO Token Clipping Exposure on Cached Batch')
    ax.set_xticks(x)
    ax.set_xticklabels([f"ε={e}" for e in eps_vals])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


## 4. Research Question 1 Analysis & Takeaways
- How does $\epsilon = 0.05$ restrict update geometry compared to $\epsilon = 0.50$?
- Did conservative clipping (small $\epsilon$) provide superior optimization stability?
